### Data Preprocessing Pipeline for Image Segmentation
- We are developing a simple data preprocessing pipeline using OpenCV and PyTorch for image segmentation tasks. 
- This pipeline includes loading an image and its corresponding mask, resizing, normalization, augmentation (random horizontal flip), and converting to PyTorch tensors.

#### 1. Import libraries

In [ ]:
import cv2
import numpy as np
import torch
from torchvision import transforms

#### 2. Load image and mask

In [ ]:
def load_image_and_mask(image_path, mask_path):
    # Load image in BGR, convert to RGB
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Load mask in grayscale (assumed: 0 = background, >0 = subject)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    return image, mask

# Example usage
image_path = "data/images/train_001.jpg"
mask_path  = "data/masks/train_001.png"

image, mask = load_image_and_mask(image_path, mask_path)
print(f"Image shape: {image.shape}, Mask shape: {mask.shape}")

- `cv2.imread(image_path)` loads the image in BGR format (OpenCV default).

- `cv2.cvtColor(..., cv2.COLOR_BGR2RGB)` converts it to RGB so it matches the usual PyTorch convention.

- `cv2.imread(..., cv2.IMREAD_GRAYSCALE)` loads the mask as a single-channel grayscale image (0 for background, non-zero for subject).

#### 3. Resize image and mask

In [ ]:
def resize_image_and_mask(image, mask, target_size=(256, 256)):
    # Resize image using bilinear interpolation
    image_resized = cv2.resize(image, target_size, interpolation=cv2.INTER_LINEAR)
    
    # Resize mask using nearest neighbor (to avoid blurring class boundaries)
    mask_resized = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)
    
    return image_resized, mask_resized

# Example usage
target_size = (256, 256)
image_resized, mask_resized = resize_image_and_mask(image, mask, target_size)
print(f"Resized image shape: {image_resized.shape}, mask shape: {mask_resized.shape}")

- `cv2.INTER_LINEAR` is good for images (smooth resizing).

- `cv2.INTER_NEAREST` is used for masks to preserve sharp boundaries (no interpolation between classes).

- `target_size` is usually a tuple like `(H, W)` or `(W, H)`; OpenCV uses `(width, height)`.

#### 4. Normalize image to and convert to float

In [ ]:
def normalize_image(image):
    # Convert to float32 and scale to [0, 1]
    image_float = image.astype(np.float32) / 255.0
    return image_float

# Example usage
image_normalized = normalize_image(image_resized)
print(f"Image dtype: {image_normalized.dtype}, range: [{image_normalized.min():.3f}, {image_normalized.max():.3f}]")

- Converts the image from uint8 (0–255) to float32 (0.0–1.0).

- This is needed before applying PyTorch normalization transforms.

#### 5. Apply random horizontal flip (augmentation)

In [ ]:
def random_horizontal_flip(image, mask, p=0.5):
    if np.random.rand() < p:
        image = np.flip(image, axis=1)
        mask  = np.flip(mask,  axis=1)
    return image, mask

# Example usage
image_aug, mask_aug = random_horizontal_flip(image_normalized, mask_resized, p=0.5)

- Flips both image and mask horizontally with probability p.

- This is a simple data augmentation that helps the model generalize better.

- Always flip the mask in the same way as the image so the labels stay aligned.

#### 6. Convert to PyTorch tensors

In [ ]:
def to_tensor(image, mask):
    # Convert image: HWC → CHW and to tensor
    image_tensor = torch.from_numpy(image).permute(2, 0, 1)  # HWC → CHW
    mask_tensor  = torch.from_numpy(mask).long()              # mask as long tensor
    
    return image_tensor, mask_tensor

# Example usage
image_tensor, mask_tensor = to_tensor(image_aug, mask_aug)
print(f"Image tensor shape: {image_tensor.shape}, mask tensor shape: {mask_tensor.shape}")

- `permute(2, 0, 1)` changes the order from `(H, W, C)` to `(C, H, W)` as expected by PyTorch models.

- `mask` is converted to `long` (int64) because segmentation masks are class indices (not floats).

#### 7. Normalize image using ImageNet stats

In [ ]:
# Define normalization transform (ImageNet mean/std)
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# Apply normalization
image_normalized_tensor = normalize(image_tensor)

- Normalizes each channel using ImageNet statistics (common for models like ResNet, VGG, etc.).

- This helps the model train faster and more stably.

- Only applied to the image; the mask is left as-is.

#### 8. Convert mask to binary (subject vs background)

In [ ]:
def make_binary_mask(mask_tensor):
    # Convert any non-zero value to 1 (subject), 0 remains background
    binary_mask = (mask_tensor > 0).long()
    return binary_mask

# Example usage
binary_mask = make_binary_mask(mask_tensor)
print(f"Unique values in mask: {torch.unique(binary_mask)}")  # Should be tensor([0, 1])

- Converts a multi-class mask into a binary mask:

  - `0` → background

  - `>0` → subject (set to 1)

- Useful if the original mask has multiple object classes but the task is just “subject vs background”.


#### Task: Implement the full pipeline and execute it.

In [4]:
import cv2
import numpy as np
import torch
from torchvision import transforms
from pathlib import Path

class ImageSegmentationPreprocessor:
    def __init__(self, target_size=(256, 256), flip_prob=0.5):
        self.target_size = target_size
        self.flip_prob = flip_prob
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                              std=[0.229, 0.224, 0.225])

    def load_image_and_mask(self, image_path, mask_path):
        """Load image and mask with error handling"""
        image_path = Path(image_path)
        mask_path = Path(mask_path)
        
        # Check if files exist
        if not image_path.exists():
            raise FileNotFoundError(f"❌ Image not found: {image_path}")
        if not mask_path.exists():
            raise FileNotFoundError(f"❌ Mask not found: {mask_path}")
        
        # Read image in BGR format, convert to RGB
        image = cv2.imread(str(image_path))
        if image is None:
            raise ValueError(f"❌ Failed to load image: {image_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Read mask in grayscale
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise ValueError(f"❌ Failed to load mask: {mask_path}")
        
        print(f"✅ Loaded: {image_path.name} | Shape: {image.shape}")
        print(f"✅ Loaded: {mask_path.name} | Shape: {mask.shape}")
        
        return image, mask

    def preprocess(self, image, mask):
        # Resize image and mask
        image = cv2.resize(image, self.target_size, interpolation=cv2.INTER_LINEAR)
        mask  = cv2.resize(mask,  self.target_size, interpolation=cv2.INTER_NEAREST)

        # Normalize image pixels to [0, 1]
        image = image.astype(np.float32) / 255.0

        # Data augmentation: random horizontal flip
        if np.random.rand() < self.flip_prob:
            image = np.flip(image, axis=1)
            mask  = np.flip(mask,  axis=1)

            # IMPORTANT: make a copy to remove negative strides
            image = image.copy()
            mask  = mask.copy()

        # Ensure contiguous (safe even if not flipped)
        image = np.ascontiguousarray(image)
        mask  = np.ascontiguousarray(mask)

        # Convert to PyTorch tensors and normalize image
        image = torch.from_numpy(image).permute(2, 0, 1)  # HWC -> CHW
        image = self.normalize(image)
        mask  = torch.from_numpy(mask).long()

        # Convert mask to binary (subject=1, background=0)
        mask = (mask > 0).long()

        return image, mask



# ============== DEBUG AND VERIFY PATHS ==============
from pathlib import Path

base_dir = Path(r"C:\Users\win10\Desktop\Batch_2\image_sample")

print("🔍 CHECKING FOLDER STRUCTURE:")
print(f"Base dir: {base_dir}")
print(f"Exists: {base_dir.exists()}\n")

# List available folders
if base_dir.exists():
    for folder in base_dir.iterdir():
        if folder.is_dir():
            file_count = len(list(folder.glob("*.*")))
            print(f"  📁 {folder.name}/ ({file_count} files)")

print("\n" + "="*60)
print("🔍 FINDING MATCHING IMAGE-MASK PAIRS:")
print("="*60)

images_dir = base_dir / "train2017"
masks_dir = base_dir / "masks_train2017"  # Adjust if different

# List images and masks
if images_dir.exists():
    images = sorted(list(images_dir.glob("*.jpg")))
    print(f"Found {len(images)} images in {images_dir.name}/")
    for img in images[:5]:
        print(f"  📸 {img.name}")
else:
    print(f"❌ Images directory not found: {images_dir}")

if masks_dir.exists():
    masks = sorted(list(masks_dir.glob("*.png")))
    print(f"\nFound {len(masks)} masks in {masks_dir.name}/")
    for msk in masks[:5]:
        print(f"  🎭 {msk.name}")
else:
    print(f"❌ Masks directory not found: {masks_dir}")

# ============== RUN PIPELINE WITH FIRST VALID PAIR ==============
print("\n" + "="*60)
print("🚀 TESTING PIPELINE:")
print("="*60)

preprocessor = ImageSegmentationPreprocessor(target_size=(256, 256))

if images_dir.exists() and masks_dir.exists():
    images = list(images_dir.glob("*.jpg"))
    
    if images:
        img_file = images[0]
        # Try to find matching mask (with or without suffix)
        mask_candidates = [
            masks_dir / img_file.name.replace('.jpg', '.png'),
            masks_dir / (img_file.stem + '_mask.png'),
        ]
        
        mask_file = None
        for candidate in mask_candidates:
            if candidate.exists():
                mask_file = candidate
                break
        
        if mask_file:
            try:
                print(f"\nProcessing:")
                print(f"  Image: {img_file}")
                print(f"  Mask:  {mask_file}\n")
                
                image, mask = preprocessor.load_image_and_mask(str(img_file), str(mask_file))
                image_tensor, mask_tensor = preprocessor.preprocess(image, mask)
                
                print(f"\n✅ PREPROCESSING SUCCESS!")
                print(f"  Image tensor shape: {image_tensor.shape}")
                print(f"  Mask tensor shape:  {mask_tensor.shape}")
                print(f"  Mask unique values: {torch.unique(mask_tensor).tolist()}")
                print(f"  Image range: [{image_tensor.min():.3f}, {image_tensor.max():.3f}]")
                
            except Exception as e:
                print(f"❌ ERROR: {e}")
        else:
            print(f"❌ No matching mask found for {img_file.name}")
            print(f"   Tried: {[c.name for c in mask_candidates]}")
    else:
        print("❌ No images found in images directory")
else:
    print("❌ Images or masks directory not found")


🔍 CHECKING FOLDER STRUCTURE:
Base dir: C:\Users\win10\Desktop\Batch_2\image_sample
Exists: True

  📁 annotations/ (6 files)
  📁 masks/ (0 files)
  📁 masks_train2017/ (118287 files)
  📁 test2017/ (4067 files)
  📁 train2017/ (11829 files)
  📁 train2017_masks/ (0 files)
  📁 val2017/ (501 files)

🔍 FINDING MATCHING IMAGE-MASK PAIRS:
Found 11828 images in train2017/
  📸 000000000036.jpg
  📸 000000000073.jpg
  📸 000000000089.jpg
  📸 000000000109.jpg
  📸 000000000196.jpg

Found 118287 masks in masks_train2017/
  🎭 000000000009_mask.png
  🎭 000000000025_mask.png
  🎭 000000000030_mask.png
  🎭 000000000034_mask.png
  🎭 000000000036_mask.png

🚀 TESTING PIPELINE:

Processing:
  Image: C:\Users\win10\Desktop\Batch_2\image_sample\train2017\000000000036.jpg
  Mask:  C:\Users\win10\Desktop\Batch_2\image_sample\masks_train2017\000000000036_mask.png

✅ Loaded: 000000000036.jpg | Shape: (640, 481, 3)
✅ Loaded: 000000000036_mask.png | Shape: (640, 481)

✅ PREPROCESSING SUCCESS!
  Image tensor shape: torch